<a href="https://colab.research.google.com/github/almendraapolaya/DI_Bootcamp_a/blob/main/Week_8/Day_2/Daily_challenge%20/Daily_challenge_w8_d2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Daily Challenge: Pinecone Serverless Reranking in Action
===

**Why are we doing this?**

Reranking models boost search relevance by assigning similarity scores between a query and documents, then
reordering results so the most pertinent information appears first. In contexts like healthcare, this helps clinicians
quickly access the most critical clinical notes.

**Part 1: Load Documents & Execute Reranking Model**

In [ ]:
!pip install -U -q pinecone==6.0.1 pinecone-notebooks

In [ ]:
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

In [ ]:
query = "Tell me about Apple's products"
documents = [
    "Apples are round fruits that grow on trees and come in varieties like Granny Smith and Fuji.",
    "The Apple iPhone 15 features a titanium design and the A17 Pro chip for high-end performance.",
    "Eating an apple a day is a common health tip because they are high in fiber and vitamin C.",
    "Apple recently released the Vision Pro, a spatial computer that blends digital content with the physical world.",
    "The MacBook Air with the M3 chip is known for being incredibly thin and fast for creative work."
]

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)

def show_reranked_results(query, matches):
    print(f"Query: {query}\n")
    print("-" * 50)
    for i, m in enumerate(matches):
        print(f"Rank {i+1} | Score: {m.score:.4f}")
        print(f"Document: {m.document.text}\n")

show_reranked_results(query, reranked.data)

**Part 2: Setup a Serverless Index for Medical Notes**

In [ ]:
!pip install -q pandas torch transformers

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

spec = ServerlessSpec(cloud=cloud, region=region)

index_name = 'medical-notes-index'

In [ ]:
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec
)

while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

print(f"Index '{index_name}' is ready!")

**Part 3: Load the Sample Data**

In [ ]:
import requests
import tempfile
import os
import pandas as pd

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    url = "https://raw.githubusercontent.com/pinecone-io/examples/main/docs/data/sample_notes_data.jsonl"

    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

print("Data shape:", df.shape)
display(df.head())

**Part 4: Upsert Data into the Index**

In [ ]:
index = pc.Index(name=index_name)

index.upsert_from_dataframe(df)

def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Current vector count in index: {vector_count}")
    return vector_count > 0

while not is_fresh(index):
    print("Waiting for vectors to be indexed...")
    time.sleep(5)

print("\nIndex ready!")
display(index.describe_index_stats())

**Part 5: Query & Embedding Function**

In [ ]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        model_output = model(**encoded_input)
        embeddings = model_output.last_hidden_state[0].mean(dim=0) # This was the error
        embeddings = model_output.last_hidden_state.mean(dim=1)

    return embeddings[0].tolist()

In [ ]:
question = "What are the common treatments for acute chest pain?"
query_vector = get_embedding(question)

print(f"Vector length: {len(query_vector)}")

if len(query_vector) == 384:
    results = index.query(vector=query_vector, top_k=10, include_metadata=True)
    sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

    for i, match in enumerate(sorted_matches[:3]):
        print(f"Match {i+1} (Score: {match['score']:.4f}):")

        metadata = match.get('metadata', {})

        note_content = metadata.get('text') or metadata.get('content') or str(metadata)

        print(f"Note Content: {note_content[:150]}...\n")
else:
    print("Error: Vector dimension is still not 384.")

**Part 6: Display & Rerank Clinical Notes**

In [ ]:
def show_results(question, matches):
    print(f"Question: '{question}'")
    print("\nInitial Search Results (Bi-Encoder):")
    for i, match in enumerate(matches):
        print(f"{str(i+1).rjust(4)}. ID: {match['id']}")
        print(f"      Score: {match['score']:.4f}")
        metadata = match.get('metadata', {})
        text_content = metadata.get('text') or metadata.get('content') or str(metadata)
        print(f"      Note: {text_content[:100]}...")
        print('')

show_results(question, sorted_matches)

transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

refined_query = "What are the specific clinical protocols and medications for treating acute chest pain?"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)


def show_reranked_results(question, matches):
    print(f"Refined Question: '{question}'")
    print("\n--- FINAL RERANKED RESULTS (Cross-Encoder) ---")
    for i, match in enumerate(matches):
        print(f"{str(i+1).rjust(4)}. ID: {match.document.id}")
        print(f"      New Rerank Score: {match.score:.4f}")
        print(f"      Clinical Data: {match.document.reranking_field[:200]}...\n")

show_reranked_results(refined_query, reranked_results.data)